# Reinforcement Learning: Visual Guide

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Digital-AI-Finance/methods-algorithms/blob/master/notebooks/L06_rl.ipynb)

**Learning Objectives:**
- Understand the gridworld environment as a Markov Decision Process
- Implement Q-learning with epsilon-greedy exploration
- Visualize learned policies and Q-value landscapes
- Analyze the effects of exploration rate and discount factor on learning

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap

np.random.seed(42)

plt.rcParams.update({
    'figure.figsize': (10, 6),
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
})

# ML color palette
ML_PURPLE = '#3333B2'
ML_BLUE = '#0066CC'
ML_ORANGE = '#FF7F0E'
ML_GREEN = '#2CA02C'
ML_RED = '#D62728'
ML_LAVENDER = '#ADADE0'

# Gridworld parameters
GRID_SIZE = 5
START = (0, 0)
GOAL = (4, 4)
WALLS = [(1, 1), (2, 3)]
TRAPS = [(3, 1), (1, 3)]

# Rewards
GOAL_REWARD = 10.0
TRAP_REWARD = -5.0
STEP_REWARD = -0.1

# Actions: 0=up, 1=right, 2=down, 3=left
ACTIONS = [(-1, 0), (0, 1), (1, 0), (0, -1)]
ACTION_NAMES = ['Up', 'Right', 'Down', 'Left']
N_ACTIONS = 4

def state_to_idx(r, c):
    return r * GRID_SIZE + c

def idx_to_state(idx):
    return idx // GRID_SIZE, idx % GRID_SIZE

def step(state_idx, action):
    r, c = idx_to_state(state_idx)
    dr, dc = ACTIONS[action]
    nr, nc = r + dr, c + dc
    # Boundary check
    if nr < 0 or nr >= GRID_SIZE or nc < 0 or nc >= GRID_SIZE:
        nr, nc = r, c  # stay in place
    # Wall check
    if (nr, nc) in WALLS:
        nr, nc = r, c  # stay in place
    next_idx = state_to_idx(nr, nc)
    # Rewards
    if (nr, nc) == GOAL:
        return next_idx, GOAL_REWARD, True
    elif (nr, nc) in TRAPS:
        return next_idx, TRAP_REWARD, True
    else:
        return next_idx, STEP_REWARD, False

print('Gridworld environment defined.')
print(f'Grid: {GRID_SIZE}x{GRID_SIZE}, Start: {START}, Goal: {GOAL}')
print(f'Walls: {WALLS}, Traps: {TRAPS}')
print(f'Rewards -- Goal: {GOAL_REWARD}, Trap: {TRAP_REWARD}, Step: {STEP_REWARD}')

## 1. The Gridworld Environment

In [ ]:
def draw_grid(ax, trajectory=None, title='5x5 Gridworld'):
    """Draw the gridworld with colored cells and labels."""
    ax.set_xlim(-0.5, GRID_SIZE - 0.5)
    ax.set_ylim(-0.5, GRID_SIZE - 0.5)
    ax.set_aspect('equal')
    ax.invert_yaxis()

    for r in range(GRID_SIZE):
        for c in range(GRID_SIZE):
            color = 'white'
            label = ''
            if (r, c) == START:
                color = '#E8F5E9'  # light green
                label = 'START'
            elif (r, c) == GOAL:
                color = '#C8E6C9'  # green
                label = f'GOAL\n+{GOAL_REWARD:.0f}'
            elif (r, c) in WALLS:
                color = '#424242'  # dark gray
                label = 'WALL'
            elif (r, c) in TRAPS:
                color = '#FFCDD2'  # light red
                label = f'TRAP\n{TRAP_REWARD:.0f}'

            rect = plt.Rectangle((c - 0.5, r - 0.5), 1, 1,
                                 facecolor=color, edgecolor='gray', linewidth=1.5)
            ax.add_patch(rect)
            if label:
                txt_color = 'white' if (r, c) in WALLS else 'black'
                ax.text(c, r, label, ha='center', va='center',
                        fontsize=8, fontweight='bold', color=txt_color)

    # Draw trajectory if provided
    if trajectory is not None and len(trajectory) > 1:
        for i in range(len(trajectory) - 1):
            r1, c1 = trajectory[i]
            r2, c2 = trajectory[i + 1]
            ax.annotate('', xy=(c2, r2), xytext=(c1, r1),
                        arrowprops=dict(arrowstyle='->', color=ML_BLUE,
                                       linewidth=2, alpha=0.7))
        # Mark start and end
        ax.plot(trajectory[0][1], trajectory[0][0], 'o', color=ML_GREEN,
                markersize=12, zorder=5)
        ax.plot(trajectory[-1][1], trajectory[-1][0], 's', color=ML_RED,
                markersize=12, zorder=5)

    ax.set_xticks(range(GRID_SIZE))
    ax.set_yticks(range(GRID_SIZE))
    ax.set_title(title)
    ax.grid(True, alpha=0.3)

# Visual 1: Gridworld layout
fig, ax = plt.subplots(figsize=(7, 7))
draw_grid(ax, title='5x5 Gridworld Environment')
plt.tight_layout()
plt.show()

## 2. Random Policy

In [ ]:
def run_episode(Q=None, epsilon=1.0, max_steps=100):
    """Run one episode. If Q is None or epsilon=1.0, acts randomly."""
    s = state_to_idx(*START)
    trajectory = [START]
    total_reward = 0
    for _ in range(max_steps):
        if Q is None or np.random.rand() < epsilon:
            a = np.random.randint(N_ACTIONS)
        else:
            a = np.argmax(Q[s])
        s_next, r, done = step(s, a)
        total_reward += r
        trajectory.append(idx_to_state(s_next))
        s = s_next
        if done:
            break
    return trajectory, total_reward

# Run 100 random episodes
random_rewards = []
sample_traj = None
for ep in range(100):
    traj, rew = run_episode(Q=None, epsilon=1.0)
    random_rewards.append(rew)
    if ep == 0:
        sample_traj = traj

# Visual 2: Trajectory + reward histogram
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: one trajectory
draw_grid(axes[0], trajectory=sample_traj, title='Random Policy: Sample Trajectory')

# Right: reward histogram
ax = axes[1]
ax.hist(random_rewards, bins=20, color=ML_PURPLE, alpha=0.7, edgecolor='white')
ax.axvline(np.mean(random_rewards), color=ML_ORANGE, linewidth=2,
           linestyle='--', label=f'Mean: {np.mean(random_rewards):.1f}')
ax.set_xlabel('Cumulative Reward')
ax.set_ylabel('Frequency')
ax.set_title('Random Policy: Reward Distribution (100 Episodes)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Random policy mean reward: {np.mean(random_rewards):.2f}')
print(f'Random policy std reward: {np.std(random_rewards):.2f}')

## 3. Q-Learning Implementation

In [ ]:
def q_learning(alpha=0.1, gamma=0.99, epsilon=0.1, n_episodes=500,
               max_steps=100, decay_epsilon=False):
    """Train Q-learning agent. Returns Q-table and reward history."""
    n_states = GRID_SIZE * GRID_SIZE
    Q = np.zeros((n_states, N_ACTIONS))
    rewards = []

    for ep in range(n_episodes):
        s = state_to_idx(*START)
        total_reward = 0
        eps = epsilon
        if decay_epsilon:
            eps = max(0.01, epsilon * (1 - ep / n_episodes))

        for _ in range(max_steps):
            # Epsilon-greedy action selection
            if np.random.rand() < eps:
                a = np.random.randint(N_ACTIONS)
            else:
                a = np.argmax(Q[s])

            s_next, r, done = step(s, a)
            total_reward += r

            # Q-learning update
            Q[s, a] += alpha * (r + gamma * np.max(Q[s_next]) - Q[s, a])
            s = s_next

            if done:
                break

        rewards.append(total_reward)

    return Q, rewards

# Train with default parameters
Q_trained, episode_rewards = q_learning(alpha=0.1, gamma=0.99, epsilon=0.1, n_episodes=500)

print(f'Q-table shape: {Q_trained.shape}  (states x actions)')
print(f'Final 50-episode mean reward: {np.mean(episode_rewards[-50:]):.2f}')

In [ ]:
# Visual 3: Learning curve with moving average
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(episode_rewards, alpha=0.3, color=ML_PURPLE, linewidth=0.8, label='Per episode')

# 50-episode moving average
window = 50
if len(episode_rewards) >= window:
    moving_avg = np.convolve(episode_rewards, np.ones(window)/window, mode='valid')
    ax.plot(range(window - 1, len(episode_rewards)), moving_avg,
            color=ML_ORANGE, linewidth=2.5, label=f'{window}-episode moving average')

ax.set_xlabel('Episode')
ax.set_ylabel('Cumulative Reward')
ax.set_title('Q-Learning: Reward per Episode')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Visualizing the Learned Policy

In [ ]:
# Visual 4: Policy arrows on grid
fig, ax = plt.subplots(figsize=(7, 7))
draw_grid(ax, title='Learned Policy (Optimal Actions)')

# Draw arrows for best action in each non-terminal, non-wall state
arrow_dx = [0, 0.3, 0, -0.3]   # up, right, down, left (in plot coords)
arrow_dy = [-0.3, 0, 0.3, 0]

for r in range(GRID_SIZE):
    for c in range(GRID_SIZE):
        if (r, c) in WALLS or (r, c) == GOAL or (r, c) in TRAPS:
            continue
        s = state_to_idx(r, c)
        best_a = np.argmax(Q_trained[s])
        ax.annotate('', xy=(c + arrow_dx[best_a], r + arrow_dy[best_a]),
                    xytext=(c, r),
                    arrowprops=dict(arrowstyle='->', color=ML_PURPLE,
                                   linewidth=2.5))

plt.tight_layout()
plt.show()

In [ ]:
# Visual 5: Q-value heatmap
max_q = np.max(Q_trained, axis=1).reshape(GRID_SIZE, GRID_SIZE)

# Mask walls
max_q_masked = max_q.copy()
for (wr, wc) in WALLS:
    max_q_masked[wr, wc] = np.nan

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(max_q_masked, cmap='RdYlGn', aspect='equal')

for r in range(GRID_SIZE):
    for c in range(GRID_SIZE):
        if (r, c) in WALLS:
            ax.text(c, r, 'WALL', ha='center', va='center',
                    fontsize=9, fontweight='bold', color='gray')
        else:
            ax.text(c, r, f'{max_q[r, c]:.1f}', ha='center', va='center',
                    fontsize=10, fontweight='bold')

ax.set_xticks(range(GRID_SIZE))
ax.set_yticks(range(GRID_SIZE))
ax.set_title('Max Q-Value per State')
plt.colorbar(im, ax=ax, label='Max Q(s,a)', shrink=0.8)
plt.tight_layout()
plt.show()

## 5. Exploration vs Exploitation

In [ ]:
# Visual 6: Compare different epsilon values
epsilons = [0.0, 0.1, 0.3, 0.5]
colors_eps = [ML_RED, ML_PURPLE, ML_BLUE, ML_ORANGE]
window = 50

fig, ax = plt.subplots(figsize=(10, 6))
for eps, color in zip(epsilons, colors_eps):
    np.random.seed(42)
    _, rewards_eps = q_learning(alpha=0.1, gamma=0.99, epsilon=eps, n_episodes=500)
    if len(rewards_eps) >= window:
        ma = np.convolve(rewards_eps, np.ones(window)/window, mode='valid')
        ax.plot(range(window - 1, len(rewards_eps)), ma,
                color=color, linewidth=2, label=f'epsilon={eps}')

ax.set_xlabel('Episode')
ax.set_ylabel(f'Reward ({window}-ep moving avg)')
ax.set_title('Exploration vs Exploitation: Effect of Epsilon')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Discount Factor Effect

In [ ]:
# Visual 7: Q-value heatmaps for different gamma values
gammas = [0.5, 0.9, 0.99]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, gamma_val in zip(axes, gammas):
    np.random.seed(42)
    Q_g, _ = q_learning(alpha=0.1, gamma=gamma_val, epsilon=0.1, n_episodes=500)
    max_q_g = np.max(Q_g, axis=1).reshape(GRID_SIZE, GRID_SIZE)
    max_q_g_masked = max_q_g.copy()
    for (wr, wc) in WALLS:
        max_q_g_masked[wr, wc] = np.nan

    im = ax.imshow(max_q_g_masked, cmap='RdYlGn', aspect='equal')
    for r in range(GRID_SIZE):
        for c in range(GRID_SIZE):
            if (r, c) in WALLS:
                ax.text(c, r, 'W', ha='center', va='center',
                        fontsize=9, color='gray', fontweight='bold')
            else:
                ax.text(c, r, f'{max_q_g[r, c]:.1f}', ha='center', va='center',
                        fontsize=9, fontweight='bold')
    ax.set_xticks(range(GRID_SIZE))
    ax.set_yticks(range(GRID_SIZE))
    ax.set_title(f'gamma = {gamma_val}')
    plt.colorbar(im, ax=ax, shrink=0.8)

fig.suptitle('Effect of Discount Factor on Q-Values', fontsize=14)
plt.tight_layout()
plt.show()

## 7. Decaying Epsilon Schedule

In [ ]:
# Visual 8: Fixed vs decaying epsilon
window = 50

np.random.seed(42)
_, rewards_fixed = q_learning(alpha=0.1, gamma=0.99, epsilon=0.3,
                              n_episodes=500, decay_epsilon=False)
np.random.seed(42)
_, rewards_decay = q_learning(alpha=0.1, gamma=0.99, epsilon=1.0,
                              n_episodes=500, decay_epsilon=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: epsilon schedule
ax = axes[0]
eps_fixed = np.full(500, 0.3)
eps_decay = np.maximum(0.01, 1.0 * (1 - np.arange(500) / 500))
ax.plot(eps_fixed, color=ML_PURPLE, linewidth=2, label='Fixed (eps=0.3)')
ax.plot(eps_decay, color=ML_ORANGE, linewidth=2, label='Decaying (1.0 -> 0.01)')
ax.set_xlabel('Episode')
ax.set_ylabel('Epsilon')
ax.set_title('Epsilon Schedule')
ax.legend()
ax.grid(True, alpha=0.3)

# Right: learning curves
ax = axes[1]
if len(rewards_fixed) >= window:
    ma_fixed = np.convolve(rewards_fixed, np.ones(window)/window, mode='valid')
    ax.plot(range(window - 1, len(rewards_fixed)), ma_fixed,
            color=ML_PURPLE, linewidth=2, label='Fixed (eps=0.3)')
if len(rewards_decay) >= window:
    ma_decay = np.convolve(rewards_decay, np.ones(window)/window, mode='valid')
    ax.plot(range(window - 1, len(rewards_decay)), ma_decay,
            color=ML_ORANGE, linewidth=2, label='Decaying epsilon')
ax.set_xlabel('Episode')
ax.set_ylabel(f'Reward ({window}-ep moving avg)')
ax.set_title('Learning Curves: Fixed vs Decaying Epsilon')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Summary

**Key Takeaways:**
- **Q-learning converges to the optimal policy** on small gridworlds with sufficient exploration -- the agent learns to navigate around walls and avoid traps
- **Epsilon controls exploration vs exploitation** -- too low means the agent gets stuck, too high means it never converges; decaying epsilon gives the best of both
- **The discount factor shapes the Q-value landscape** -- high gamma (0.99) makes distant rewards visible, low gamma (0.5) makes the agent myopic
- **Policy and Q-value visualizations are essential** for debugging RL agents -- always inspect what the agent learned, not just its reward curve